# Avro Basics — 01: Reading Avro Files

## What you will learn
Avro is the standard format for event streaming pipelines (Kafka, Pulsar).
Unlike Parquet (columnar/analytics), Avro is **row-based and schema-first**,
optimised for serialization and schema evolution.

In this notebook:
1. What Avro is and when to use it vs Parquet
2. Reading Avro files with `format("avro")`
3. Schema — how Avro embeds it in the file header
4. avroSchema option and reading multiple files
5. recursiveFileLookup and input_file_name()
6. Verifying the spark-avro JAR is loaded


In [1]:
import os, time, pathlib, shutil, random, datetime, json as pyjson
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/avro_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('avro-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | Avro JAR: spark-avro_2.13-4.0.2")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/10 12:19:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.2 | Avro JAR: spark-avro_2.13-4.0.2


In [2]:
# Generate sample Avro files
import pathlib, random, datetime
random.seed(42)

AVRO_DIR = f"{DATA_DIR}/avro_files"
pathlib.Path(AVRO_DIR).mkdir(exist_ok=True)

from pyspark.sql.types import *

schema = StructType([
    StructField("event_id",   LongType()),
    StructField("user_id",    LongType()),
    StructField("event_type", StringType()),
    StructField("page",       StringType()),
    StructField("revenue",    DoubleType()),
    StructField("event_ts",   TimestampType()),
])

EVENTS = ["page_view","click","purchase","search","add_to_cart"]
PAGES  = ["/home","/products","/cart","/checkout"]

rows = []
base_ts = datetime.datetime(2024, 1, 1, 0, 0, 0)
for i in range(10_000):
    ts = base_ts + datetime.timedelta(seconds=i*10)
    rows.append((i+1, random.randint(1,500), random.choice(EVENTS),
                 random.choice(PAGES),
                 round(random.uniform(10,500),2) if random.random()>0.7 else 0.0,
                 ts))

df_events = spark.createDataFrame(rows, schema)
df_events.coalesce(2).write.format("avro").mode("overwrite").save(AVRO_DIR)
print(f"Written {df_events.count():,} events as Avro to {AVRO_DIR}")

import glob as glib
avro_files = glib.glob(f"{AVRO_DIR}/*.avro")
print(f"Files: {len(avro_files)}")
for f in avro_files:
    print(f"  {pathlib.Path(f).name}: {pathlib.Path(f).stat().st_size//1024} KB")

Written 10,000 events as Avro to /workspace/data/avro_basics/avro_files
Files: 2
  part-00000-484aaaf6-1f03-4c01-8c2a-8e0267cf0023-c000.snappy.avro: 93 KB
  part-00001-484aaaf6-1f03-4c01-8c2a-8e0267cf0023-c000.snappy.avro: 89 KB


## Step 1 — Basic Read


In [3]:
# Read Avro directory
df = spark.read.format("avro").load(AVRO_DIR)
print(f"Rows: {df.count():,}")
df.printSchema()
df.show(5)

Rows: 10,000
root
 |-- event_id: long (nullable = true)
 |-- user_id: long (nullable = true)
 |-- event_type: string (nullable = true)
 |-- page: string (nullable = true)
 |-- revenue: double (nullable = true)
 |-- event_ts: timestamp (nullable = true)

+--------+-------+-----------+---------+-------+-------------------+
|event_id|user_id| event_type|     page|revenue|           event_ts|
+--------+-------+-----------+---------+-------+-------------------+
|       1|    328|  page_view|    /home|  130.0|2024-01-01 00:00:00|
|       2|     72|  page_view|    /home|    0.0|2024-01-01 00:00:10|
|       3|     17|  page_view|    /home|    0.0|2024-01-01 00:00:20|
|       4|    259|add_to_cart|    /home|    0.0|2024-01-01 00:00:30|
|       5|    367|add_to_cart|/checkout|    0.0|2024-01-01 00:00:40|
+--------+-------+-----------+---------+-------+-------------------+
only showing top 5 rows


## Step 2 — Avro Schema Inspection

Avro embeds its schema as JSON in the file header.
Let's read it with pyarrow to understand the format.


In [4]:
# Avro stores its schema as JSON in every file header.
# Use fastavro to read it directly from the binary file.
import glob as glib

avro_files = glib.glob(f"{AVRO_DIR}/*.avro")
avro_file  = avro_files[0]

# Method 1: fastavro (most direct — reads Avro OCF header)
try:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                    "--break-system-packages", "fastavro"], check=True)
    import fastavro
    with open(avro_file, "rb") as f:
        reader = fastavro.reader(f)
        schema = reader.writer_schema
    print("Avro schema (from file header via fastavro):")
    import json as pyjson
    print(pyjson.dumps(schema, indent=2)[:800])
except Exception as e:
    print(f"fastavro not available: {e}")

# Method 2: Spark always infers schema from Avro header automatically
print("\nSpark-inferred schema (same source — Avro header):")
spark.read.format("avro").load(AVRO_DIR).printSchema()
print("Spark reads the Avro schema from the file header automatically.")
print("No schema argument needed when reading — unlike CSV.")

fastavro not available: Command '['/usr/bin/python3', '-m', 'pip', 'install', '--quiet', '--break-system-packages', 'fastavro']' returned non-zero exit status 2.

Spark-inferred schema (same source — Avro header):
root
 |-- event_id: long (nullable = true)
 |-- user_id: long (nullable = true)
 |-- event_type: string (nullable = true)
 |-- page: string (nullable = true)
 |-- revenue: double (nullable = true)
 |-- event_ts: timestamp (nullable = true)

Spark reads the Avro schema from the file header automatically.
No schema argument needed when reading — unlike CSV.



Usage:   
  /usr/bin/python3 -m pip install [options] <requirement specifier> [package-index-options] ...
  /usr/bin/python3 -m pip install [options] -r <requirements file> [package-index-options] ...
  /usr/bin/python3 -m pip install [options] [-e] <vcs project url> ...
  /usr/bin/python3 -m pip install [options] [-e] <local project path> ...
  /usr/bin/python3 -m pip install [options] <archive url/path> ...

no such option: --break-system-packages


## Step 3 — avroSchema Option: Reader Schema


In [5]:
# Read with explicit reader schema (subset of columns)
reader_schema = """
{
  "type": "record",
  "name": "Event",
  "fields": [
    {"name": "event_id",   "type": "long"},
    {"name": "user_id",    "type": "long"},
    {"name": "event_type", "type": "string"},
    {"name": "revenue",    "type": "double"}
  ]
}
"""

df_subset = spark.read.format("avro") \
                 .option("avroSchema", reader_schema) \
                 .load(AVRO_DIR)
print("Reading only 4 columns with explicit avroSchema:")
df_subset.show(5)
print(f"\nNote: Avro does NOT support column pruning like Parquet")
print("All bytes are read; only specified fields are returned to Spark")

Reading only 4 columns with explicit avroSchema:
+--------+-------+-----------+-------+
|event_id|user_id| event_type|revenue|
+--------+-------+-----------+-------+
|       1|    328|  page_view|  130.0|
|       2|     72|  page_view|    0.0|
|       3|     17|  page_view|    0.0|
|       4|    259|add_to_cart|    0.0|
|       5|    367|add_to_cart|    0.0|
+--------+-------+-----------+-------+
only showing top 5 rows

Note: Avro does NOT support column pruning like Parquet
All bytes are read; only specified fields are returned to Spark


## Step 4 — Reading Multiple Files


In [6]:
# 1. Read whole directory (recommended — Spark lists files internally)
df_dir = spark.read.format("avro").load(AVRO_DIR)
print(f"Directory read : {df_dir.count():,} rows")

# 2. Explicit list of files (resolved by Python glob — always works on local FS)
files = glib.glob(f"{AVRO_DIR}/*.avro")
df_list = spark.read.format("avro").load(files)
print(f"List of {len(files)} files: {df_list.count():,} rows")

# 3. Two specific files only
df_two = spark.read.format("avro").load(files[:2])
print(f"First 2 files  : {df_two.count():,} rows")

# 4. recursiveFileLookup — scans subdirectories too
pathlib.Path(f"{AVRO_DIR}/subdir").mkdir(exist_ok=True)
df_events.limit(100).coalesce(1).write.format("avro").mode("overwrite") \
         .save(f"{AVRO_DIR}/subdir")
df_recursive = spark.read.format("avro") \
                    .option("recursiveFileLookup", "true") \
                    .load(AVRO_DIR)
print(f"recursiveFileLookup: {df_recursive.count():,} rows (includes subdir/)")

# 5. Track source filename per row
df_src = spark.read.format("avro").load(AVRO_DIR) \
              .withColumn("_source_file", F.input_file_name())
df_src.select("event_id", "_source_file").show(4, truncate=False)

print()
print("Notes on load() path argument:")
print("  .load(AVRO_DIR)        → reads all files in directory  ✅")
print("  .load(list_of_paths)   → reads specific files          ✅")
print("  .load('/path/*.avro')  → glob string (NOT reliable on local FS — avoid)")
print("  .load(*files)          → broken (first arg = format name)")

Directory read : 10,000 rows
List of 2 files: 10,000 rows
First 2 files  : 10,000 rows
recursiveFileLookup: 10,100 rows (includes subdir/)
+--------+--------------------------------------------------------------------------------------------------------------+
|event_id|_source_file                                                                                                  |
+--------+--------------------------------------------------------------------------------------------------------------+
|1       |file:///workspace/data/avro_basics/avro_files/part-00000-484aaaf6-1f03-4c01-8c2a-8e0267cf0023-c000.snappy.avro|
|2       |file:///workspace/data/avro_basics/avro_files/part-00000-484aaaf6-1f03-4c01-8c2a-8e0267cf0023-c000.snappy.avro|
|3       |file:///workspace/data/avro_basics/avro_files/part-00000-484aaaf6-1f03-4c01-8c2a-8e0267cf0023-c000.snappy.avro|
|4       |file:///workspace/data/avro_basics/avro_files/part-00000-484aaaf6-1f03-4c01-8c2a-8e0267cf0023-c000.snappy.avro|
+------